# Batch Run Traffic Signal Control Experiment

This project studies the affect of "reward sharing"/"peer rewarding" on multi-agent reinforcement learning for traffic signal control.

This notebook compares simple baselines, MAPPO, and reward-sharing MAPPO on a road network.

This notebook runs a single experiment. To run a batch with more statistics and graphs, use the `batch_run.ipynb` file.

### Set up the environment
The code can be run in Google Colab or a local IDE.

In [ ]:
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = Path("/content/marl-tsc")
    if not ROOT.exists():
        subprocess.run([
            "git", "clone", "--branch", "main-flow-breakup",
            "--single-branch", "https://github.com/abergh18/marl-tsc.git", str(ROOT),
        ], check=True)
    sys.path.insert(0, str(ROOT / "src"))
else:
    ROOT = next(
        folder for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / "src" / "marl_tsc" / "notebook_setup.py").exists()
    )
    sys.path.insert(0, str(ROOT / "src"))

from marl_tsc.notebook_setup import setup_notebook

setup_notebook(IN_COLAB, ROOT)

In [ ]:
if not IN_COLAB:
  %reload_ext autoreload
  %autoreload 2

import matplotlib.pyplot as plt

from marl_tsc.baselines import fixed_time_actions, random_actions
from marl_tsc.mappo import train_mappo
from marl_tsc.network_types import CityNetwork, GridNetwork
from marl_tsc.simulation_generator import SimulationGenerator
from marl_tsc.training import (
    evaluate_policies,
    export_policy_replay,
    evaluation_results_table,
    plot_training_histories,
)

In [ ]:
from marl_tsc.notebook_setup import enable_colab_downloads

enable_colab_downloads(IN_COLAB)

## 1. Configure and generate the simulation

In [ ]:
SEED = 42
EPISODE_STEPS = 600
SECONDS_PER_ACTION = 5
SIMULATION_DURATION = EPISODE_STEPS * SECONDS_PER_ACTION
TRAFFIC_SPAWN_DURATION = int(SIMULATION_DURATION * 1.20)
TOTAL_TIMESTEPS = 50_000
EVALUATION_EPISODES = 3
LEARNING_RATE = 3e-3

# The same environment settings are used for training, evaluation, and replay.
ENV_KWARGS = {
    "green_phase_count": None,
    "min_green_seconds": 10,
    "seconds_per_action": SECONDS_PER_ACTION,
    "switch_penalty": 0.1,
    "collect_global_metrics": True,
    "global_metric_interval": 10,
}

OUTPUT_DIR = ROOT / "outputs"
SIMULATION_DIR = OUTPUT_DIR / "simulation"
#network = GridNetwork(4)
#network = GridNetwork(5)
network = CityNetwork(city_name="The Bearpit, Bristol, UK", radius=280)
#network = CityNetwork(city_name="Camberwell Green, London, UK", radius=500)
#network = CityNetwork(city_name="Park Square, Sheffield, UK", radius=425)
generator = SimulationGenerator(
    output_dir=SIMULATION_DIR,
    network=network,
    trip_begin=0,
    trip_end=TRAFFIC_SPAWN_DURATION,
    trip_period=2,
    seed=SEED,
)

paths = generator.generate_all()
traffic_light_ids = list(paths.traffic_light_ids)
print(f"Generated SUMO config: {paths.config_file}")
print(f"Traffic-light agent count: {len(traffic_light_ids)}")

## 2. Train MAPPO policies

The reward-sharing policy has an extra sharing action during training. Both policies are evaluated using the same traffic metrics.

In [ ]:
# Use the same arguements for all MAPPO training; use_peer_reward will be set individually
MAPPO_KWARGS = {
    "config_file": paths.config_file,
    "traffic_light_ids": traffic_light_ids,
    "output_dir": OUTPUT_DIR,
    "total_timesteps": TOTAL_TIMESTEPS,
    "rollout_steps": 256,
    "max_steps": EPISODE_STEPS,
    "seed": SEED,
    "learning_rate": LEARNING_RATE,
    "env_kwargs": ENV_KWARGS,
}

mappo_model, mappo_history, mappo_model_path = train_mappo(
    **MAPPO_KWARGS,
    use_peer_reward=False,
)

In [ ]:
reward_sharing_model, reward_sharing_history, reward_sharing_model_path = train_mappo(
    **MAPPO_KWARGS,
    use_peer_reward=True,
)

In [ ]:
fig, ax = plot_training_histories({
    "MAPPO": mappo_history,
    "Reward-sharing MAPPO": reward_sharing_history,
})
plt.show()

In [ ]:
from marl_tsc.evaluate_gifting import gifting_visualisation, print_gifting_summary

print_gifting_summary(reward_sharing_history, traffic_light_ids)

In [ ]:
gifting_visualisation(
    history=reward_sharing_history,
    agent_ids=traffic_light_ids,
    algorithm_name="Reward Sharing MAPPO",
    output_dir=str(OUTPUT_DIR),
    smooth_window=20,
    save=True,
    show=True,
)

## 3. Export a SUMO replay
A replay export is useful for validating the results and viewing how a policy performs.

In [ ]:
replay_config = export_policy_replay(
    config_file=paths.config_file,
    traffic_light_ids=traffic_light_ids,
    policy=reward_sharing_model,
    output_dir=OUTPUT_DIR / "replays" / "reward_sharing_mappo",
    max_steps=EPISODE_STEPS,
    seed=SEED,
    env_kwargs=ENV_KWARGS,
)
print(f"Open this file in the SUMO GUI: {replay_config}")

## 4. Compare all policies

Every policy uses the same evaluation episodes, seeds, and environment settings.

In [ ]:
policies = {
    "Random": random_actions,
    "Fixed-Time": fixed_time_actions,
    "MAPPO": mappo_model,
    "Reward-sharing MAPPO": reward_sharing_model,
}

policy_results = evaluate_policies(
    config_file=paths.config_file,
    traffic_light_ids=traffic_light_ids,
    policies=policies,
    episodes=EVALUATION_EPISODES,
    max_steps=EPISODE_STEPS,
    seed=SEED,
    env_kwargs=ENV_KWARGS,
)

display(evaluation_results_table(policy_results))